In [1]:
import random
import json
import os
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
import asyncio

import os
from openai import AsyncOpenAI 
from agents import Agent, Runner 
from agents import set_default_openai_client
from agents import OpenAIResponsesModel 
from agents import set_tracing_disabled
set_tracing_disabled(True)#for jupyter, remove before moving to CLI
from pandas import read_csv
import copy 

import numpy as np

### Playbook definition

In [2]:
class Playbook:
    def __init__(self, jsonl_path: str, docs_root: str = "docs/playbook"):
        self.jsonl_path = Path(jsonl_path)
        self.docs_root = Path(docs_root)

        # --- Load & parse JSONL --------------------------------------------
        self.chunks: List[Dict[str, Any]] = []

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue  # skip blank lines
                try:
                    chunk = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Malformed JSON on line {line_no} of {self.jsonl_path}"
                    ) from exc
                self.chunks.append(chunk)

        # Build quick look‑ups
        self.by_id: Dict[str, Dict[str, Any]] = {c["chunk_id"]: c for c in self.chunks}
        self.tag_index: Dict[str, List[str]] = {}
        for c in self.chunks:
            for tag in c["tags"]:
                self.tag_index.setdefault(tag, []).append(c["chunk_id"])

    # ------------------------------------------------------------
    #  API helpers
    # ------------------------------------------------------------
    def get_chunks_by_tags(self, tags: List[str]) -> List[Dict[str, Any]]:
        """Return the intersection of chunks that contain *all* supplied tags."""
        if not tags:
            return []

        # Start with the set for the first tag
        common = set(self.tag_index.get(tags[0], []))
        for t in tags[1:]:
            common &= set(self.tag_index.get(t, []))

        return [self.by_id[cid] for cid in common]

    def load_markdown_for_chunk(self, chunk: Dict[str, Any]) -> str:
        """
        The `doc_id` field points at the markdown file name *without* extension.
        e.g.  `doc_id="medical_necessity.md"` → load `docs/playbook/medical_necessity.md`
        """
        doc_file = self.docs_root / f"{chunk['doc_id']}"
        if not doc_file.exists():
            return f"[ERROR: missing {doc_file}]"

        return doc_file.read_text(encoding="utf-8")

    def load_markdown_for_tags(self, tags: List[str]) -> Dict[str, str]:
        """Return a mapping of doc_id → file contents for all chunks matching the tags."""
        chunks = self.get_chunks_by_tags(tags)
        return {c["doc_id"]: self.load_markdown_for_chunk(c) for c in chunks}


# claim_playbook = Playbook('data/playbook_chunks.jsonl')

In [3]:
# ───────────────────────────────────────────────────────────────────────
#  Imports
# ───────────────────────────────────────────────────────────────────────
from datetime import datetime

# Agents‑SDK core objects
from agents.agent import Agent,ToolContext  
from agents import SQLiteSession
from agents import FunctionTool

# The Responses‑only model provider
from agents import OpenAIResponsesModel
from pydantic import BaseModel

In [4]:
# wilo: tools work, replace with model calls

In [9]:
class OllamaResponsesAgent():
    def __init__(
        self,
        *,
        ollama_base_url: str = "http://localhost:11434/v1",
        model_name: str = 'gpt-oss:20b',
        sqlite_uri: str = "sqlite:///erisa_agentic.db",
    ):

        
        """
        Initialise the agent, its memory store, and the Ollama Responses model.

        Parameters
        ----------
        ollama_base_url : str
            Base URL of the Ollama instance 
        model_name : str
            Name of the model to use inside Ollama.
        sqlite_uri : str
            URI for the SQLite session store.
        """
        # self.session = SQLiteSession("conversation_123")

        client = AsyncOpenAI(
            api_key='ollama',
            base_url=ollama_base_url,  # key detail: route requests to Ollama (local or cloud)
        )
        
        set_default_openai_client(client)

        self.model = OpenAIResponsesModel(
            model=model_name,
            openai_client=client,
        )
        
        self.tools = [
            self._predict_denial_taxonomy_tool(),
            self._retrieve_playbook_tool(),
        ]

        self.agent = Agent(
            name = "Claim Agent",
            model=self.model,
            tools=self.tools,
            instructions="""
                        You are a patient advocate. 
                        Always call the tools predict_denial_taxonomy and retrieve_playbook.
                        You will be given a claim with relevant information and your job is to suggest a recommendation for how to best proceed.
                        This recommendation can be one of the following options and nothing else: pursue, do_not_pursue, or needs_info. Always use
                        the retrieve_playbook tool to get more information and instructions specific to the type of claim and denial code.
                    
                        Afterwards, justify your recommendation.
                        """
        )
        self.playbook = Playbook('data/playbook_chunks.jsonl')
        self.tag_agent = Agent(
                name="Tag Assigning Agent",
                instructions="""
                You will be given an insurance claim and asked to assign any number of tags to it.
                If you find any of the following in the claim, it MUST be one of your tags: CO-16, CO-27, CO-29, CO-45, CO-50, CO-97
                Also assign any of the following tags if it appropriate: coding_bundling, eligibility, general, medical_necessity, missing_info, other, timely_filing, underpayment
                """,
                model=self.model,
            )
    def _predict_denial_taxonomy_tool(self) -> FunctionTool: #todo####################################################################################################
        """Return a FunctionTool that gives the current UTC time."""

        async def predict_denial_taxonomy(denial_code: str, denial_text: str) -> str:# replace with model call
            """Predict denial for claim"""
            denial_reason = ['coding_bundling', 'eligibility', 'medical_necessity','missing_info', 'other', 'timely_filing', 'underpayment']
            return denial_reason[random.randint(0,len(denial_reason)-1)]

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="predict_denial_taxonomy",
            description="predict denial reason for a claim. Provide denial code and denial text and no other parameters",
            on_invoke_tool=predict_denial_taxonomy,
            params_json_schema={
                    "type": "object",
                    "properties": {
                        # "tool_context": {"type": "string", "description": "Context for tool use"}, #specifying what claim info could help here
                        "denial_code": {"type": "string", "description": "the 4-6 digit code representing the reason the claim was denied"}, 
                        "denial_text": {"type": "string", "description": "the text describing why the claim was denied"}, 
                    },
                    "required": ["denial_code","denial_text"],
                },
        )
    def _retrieve_playbook_tool(self) -> FunctionTool: #todo: error because model gave 2 input params instead of 1. Apply to prompt###############################################################
        """Get additional instructions and context from playbook using denial code and denial reason"""

        async def retrieve_playbook(_:ToolContext,tag_prompt: str) -> str:
            """Get additional instructions and context from playbook using denial code and denial reason"""
            tags = await Runner.run(self.tag_agent,tag_prompt) 
            return self.playbook.load_markdown_for_tags(tags.final_output)

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="retrieve_playbook",
            description="Get additional instructions and context from playbook using denial code and denial reason. Provide a single string describing the claim and nothing else.",
            on_invoke_tool=retrieve_playbook,
            params_json_schema={
                    "type": "object",
                    "properties": {
                        # "tool_context": {"type": "string", "description": "Context for tool use"}, #specifying what claim info could help here
                        "claim_information": {"type": "string", "description": "the denial text for the claim"}, #specifying what claim info could help here
                    },
                    "required": ["claim_information"],
                    "additionalProperties": False,
                },
        )
    # ------------------------------------------------------------------
    #  Public API: run a single turn
    # ------------------------------------------------------------------
    async def run(self, prompt: str) -> str:
        """
        Send a prompt to the Agent and receive its response.

        The response may include tool calls; the Agent handles executing
        those tools automatically (via the FunctionTool infrastructure).

        Parameters
        ----------
        prompt : str
            The user message to send to the Agent.

        Returns
        -------
        str
            The raw text response from the Agent (after any tool calls).
        """
        result = await Runner.run(self.agent,prompt)

        return result

agent = OllamaResponsesAgent()
claims = read_csv('data/claims.csv')
for i,row in claims.iterrows():
    claim_dict = row.to_dict()
    result = await agent.run(str(claim_dict))
    print('Claim ',i,'\n',result.final_output)
    if i >= 5:
        break
            

Claim  0 
 pursue

This denial is based solely on a fee‑schedule limitation (the charge exceeds the maximum allowable amount). The payer has already paid the allowable portion ($7,200), leaving a $2,800 gap that is contestable. Because there have been no prior appeals and the claim is within an ERISA plan—where formal appeals are typically permitted—there is an opportunity to request reconsideration or to submit additional documentation (e.g., clinical justification, alternative fee schedules, or evidence of the necessity of the higher charge). Therefore, pursuing an appeal is the most constructive next step.
Claim  1 
 pursue

This denial indicates the payer applied the contracted rate and adjusted the claim per the fee schedule, resulting in a partial payment. Under the ERISA plan, the provider has the right to appeal if the contracted rate is believed to be too low or if additional documentation can support a higher reimbursement. With 18 days remaining and no prior appeals, the pro